In [236]:
import torch
from math import sqrt

In [97]:
with open(
    "Obras_completas_ilustradas_Federico_Garcia_Lorca_djvu.txt",
    "r",
    encoding="utf-8"
) as f:

    text = f.read()


In [98]:
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: [itos[i] for i in l]

In [99]:
text[:8]
encoded_text = torch.tensor(encode(text), dtype = torch.long)

In [255]:
train_size = int(len(text)*0.9)
train_data = encoded_text[:train_size]
test_data = encoded_text[train_size:]
block_size = 8
batch_size = 1
torch.manual_seed(0)

def get_batch(split):
    # generate a small batch of data of inputs x and targets y

    ix = torch.randint(len(split) - block_size, (batch_size,))
    x = torch.stack([split[i:i+block_size] for i in ix])
    y = torch.stack([split[i+1:i+block_size+1] for i in ix])
    return x, y

In [256]:
first_batch = get_batch(train_data)
first_batch[1]

tensor([[36, 47, 28,  1,  1, 46, 28, 41]])

In [397]:
vocab_size = len(chars)
d_model = 10
embedding = torch.nn.Embedding(vocab_size, d_model)

In [398]:
vocab_size

110

In [ ]:
class Attention(torch.nn.Module):

    def __init__(self):
        super().__init__()
        self.
        

In [343]:
positional_encoding = torch.empty(1,8,10)

In [346]:
positional_encoding[:,1]

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [388]:
class positionalEncoding(torch.nn.Module):

    def __init__(self, batch_size, block_size, d_model):

        self.positional_encoding = torch.tensor((batch_size, block_size, d_model))
    def encode_vector():
        for pos in range(block_size):
            for i in range(0, d_model, 2):
                denom = 10000 ** (i / d_model)
                self.positional_encoding[:, pos, i]     = math.sin(pos / denom)
                self.positional_encoding[:, pos, i + 1] = math.cos(pos / denom)
                
        return None

In [494]:

#class head(torch.nn.Module):

class decoder_only_transformer(torch.nn.Module):

    def __init__(self, vocab_size, block_size, d_model, d_k=5):

        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_model = d_model
        self.d_k = d_k
        self.embedding = torch.nn.Embedding(vocab_size, d_model)
        self.W_q = torch.nn.Linear(d_model, d_k)
        self.W_k = torch.nn.Linear(d_model, d_k)
        self.W_v = torch.nn.Linear(d_model, d_model)
        self.ln_1 = torch.nn.LayerNorm(d_model)
        self.ln_2 = torch.nn.LayerNorm(d_model)
        self.W_ff1 = torch.nn.Linear(d_model, 4*d_model)
        self.W_ff2 = torch.nn.Linear(4*d_model, d_model)
        self.linear = torch.nn.Linear(d_model, vocab_size)
        self.register_buffer("pos_encoding", self.sinusoidal_pe(block_size, d_model))
        self.register_buffer("causal_mask", self.ins_causal_mask(block_size))

    
    @staticmethod
    def sinusoidal_pe(block_size, d_model):
        pe = torch.zeros(block_size, d_model)
        for pos in range(block_size):
            for i in range(0, d_model, 2):
                denom = 10000 ** (i / d_model)
                pe[pos, i]     = math.sin(pos / denom)
                pe[pos, i + 1] = math.cos(pos / denom)
        return pe

    @staticmethod
    def ins_causal_mask(block_size):
        mask = torch.triu(torch.ones(block_size, block_size), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask
    
    def forward(self, vec_ins):
        B, T = vec_ins.shape
        X_e = self.embedding(vec_ins) + self.pos_encoding[:T]
        Q = self.W_q(X_e) #(B,T,d_k)
        K = self.W_k(X_e) #(B,T,d_k)
        V = self.W_v(X_e) #(B,T,d_model)
        QK = Q @ K.transpose(1,2)
        QK_scale = QK/math.sqrt(self.d_k) #(B,T,T)
        QK_scale += self.causal_mask[:T, :T]
        QV_softmax = torch.nn.functional.softmax(QK_scale, dim = 2) #(B,T,T)
        added_weights = QV_softmax @ V
        norm_res_att = self.ln_1(X_e + added_weights)
        linear_1 = self.W_ff1(norm_res_att)
        non_linearity = torch.nn.functional.gelu(linear_1)
        linear_2 = self.W_ff2(non_linearity)
        norm_linear = self.ln_2(linear_2 + norm_res_att)
        logits = self.linear(norm_linear)
        
        return logits


In [495]:
import torch

# Model instance
modelo = decoder_only_transformer(vocab_size, batch_size, block_size, d_model)
modelo.eval()

# Without gradients
with torch.no_grad():
    logits = modelo(first_batch[0])  # Ejecuta el forward()

    clases_predichas = torch.argmax(logits, dim=-1)

    # Opción B: Obtener las probabilidades (0.0 a 1.0) para cada clase
    probabilidades = torch.softmax(logits, dim=-1)

# Imprimir resultados
print("Logits raw:\n", logits)
print("Classes predichas:\n", decode(clases_predichas))
print("Probabilidades:\n", probabilidades)

Logits raw:
 tensor([[[ 8.4516e-01,  1.5360e-01,  3.4889e-01,  5.6551e-02, -8.1964e-01,
           3.5343e-01, -8.4621e-01, -5.1085e-01, -5.7696e-01, -3.0249e-01,
          -1.6932e-01,  2.3965e-01, -5.8747e-01, -4.1455e-01, -3.2849e-01,
          -6.1683e-01,  8.6515e-01, -1.7714e-01, -2.2778e-01, -6.4815e-01,
          -9.4265e-01, -1.1158e+00,  4.9502e-01, -9.8497e-01, -2.3350e-01,
           5.1035e-01, -1.7370e-01,  7.7159e-01,  5.0478e-01,  7.1654e-02,
          -2.9876e-01,  6.0262e-01,  8.0409e-01, -1.9964e-01, -2.5610e-01,
           2.2399e-01,  2.9205e-01, -2.8557e-01, -1.2252e+00, -1.1612e-01,
          -7.7040e-01,  1.2649e+00, -2.6023e-01, -3.5663e-01, -6.0582e-01,
          -9.5028e-01, -3.3690e-01,  1.7345e-01,  2.9981e-01, -5.0636e-01,
          -9.7639e-01, -1.6817e-01, -5.9120e-02,  1.2735e-02, -7.2146e-01,
           1.0136e+00, -7.6545e-01, -3.3259e-01,  7.3969e-01,  3.2410e-01,
          -4.1699e-01,  2.0690e-01,  2.8778e-01, -2.9261e-01, -1.1443e+00,
           1

KeyError: tensor([107,  55,  16,  95,  92,  92,  55,  95])

<method-wrapper '__repr__' of builtin_function_or_method object at 0x14008f830>

In [420]:
clases_predichas.tolist()[0]

[51, 85, 55, 48, 58, 58, 53, 62]

In [ ]:
my_first_model = decoder_only_transformer()
my_first_model(first_batch[0])

In [421]:
decode(clases_predichas.tolist()[0])

['X', '¡', '\\', 'U', 'a', 'a', 'Z', 'e']

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(first_batch[0][0], first_batch[1][0])
print(f'Total loss for this batch: {loss.item()}')

In [ ]:
vocab_size = len(chars)
embedding_dim = 10
embedding = torch.nn.Embedding(vocab_size, embedding_dim)